# Higher-quality SDXL + Refiner — 48 episode images
Paste the complete `image-prompts.md` document from Telegram. This notebook generates 48 images one at a time using **Stable Diffusion XL Base 1.0 plus the official SDXL Refiner 1.0**. It uses 1024×576 landscape images, a higher sampling-step count, and saves each completed image.

**No Python editing is needed.** In Colab choose Runtime → Change runtime type → GPU (T4 if offered). GPU availability and session duration are controlled by Google. This higher-quality two-stage setup is slower and more memory-intensive than SDXL Base alone.

No Gemini key, image API key, Cloudflare, GitHub token, or Hugging Face login is required for these public model files. It creates a ZIP for the existing GitHub import workflow. The refiner can improve fine detail, but no model can guarantee every generated image will be perfect or match a premium hosted image service.

In [ ]:
# This notebook does not use Gradio. Remove Colab's preinstalled Gradio packages to avoid their conflicting Hugging Face Hub requirement.
%pip -q uninstall -y gradio gradio-client
# Install a mutually compatible Diffusers / Transformers stack and keep Pillow below 13.
%pip -q install --upgrade "pillow>=10,<13" "huggingface-hub>=0.34,<1.0" "diffusers>=0.30,<1.0" "transformers>=4.40,<5.0" "accelerate>=0.30" "safetensors" "ipywidgets>=7.7,<9"
print('Dependencies installed. If this cell asks for a runtime restart, restart once, then continue from the GPU setup cell.')

In [ ]:
import torch, os, re, io, zipfile, shutil, time
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown
import ipywidgets as widgets

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab choose Runtime → Change runtime type → T4 GPU (if available), then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
WORK = Path('/content/science-channel-image-job')
IMAGE_DIR = WORK / 'images'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
print('Working folder:', WORK)

## 1. Paste the Telegram prompt document
Copy the entire contents of `image-prompts.md` from Telegram and paste it into this box. Keep the `## Scene 01` through `## Scene 48` headings.

In [ ]:
# This cell is self-contained: it imports the widget library even if earlier cells were skipped.
import ipywidgets as widgets
from google.colab import output
from IPython.display import display, Markdown

output.enable_custom_widget_manager()
prompt_box = widgets.Textarea(
    value='',
    placeholder='Paste the complete image-prompts.md text here…',
    description='Prompts:',
    layout=widgets.Layout(width='100%', height='360px'),
    style={'description_width': 'initial'}
)
display(prompt_box)
print('Paste all 48 scene prompts into the box above. Do not paste them into a code cell.')

In [ ]:
def parse_scene_prompts(markdown_text):
    # Accept the planner's real format: '## Scene 1: Heading', as well as
    # '## Scene 01', '### Scene 01 - Heading', and similar heading variants.
    matches = list(re.finditer(r'(?mi)^\s*#{2,6}\s*Scene\s*0*(\d+)\b[^\n]*$', markdown_text))
    prompts = {}
    for index, match in enumerate(matches):
        number = int(match.group(1))
        end = matches[index + 1].start() if index + 1 < len(matches) else len(markdown_text)
        body = markdown_text[match.end():end].strip()
        body = re.sub(r'\n{3,}', '\n\n', body)
        # Ignore markdown separators and empty lines, but retain all prompt content.
        body = re.sub(r'^\s*(?:---+|\*\*\*)\s*$', '', body, flags=re.M).strip()
        if not body:
            raise ValueError(f'Scene {number:02d} has no prompt text.')
        if number in prompts:
            raise ValueError(f'Duplicate scene heading: {number:02d}')
        prompts[number] = body
    expected = set(range(1, 49))
    if set(prompts) != expected:
        missing = sorted(expected - set(prompts))
        extra = sorted(set(prompts) - expected)
        preview = '\n'.join(markdown_text.splitlines()[:12])
        raise ValueError(
            f'Could not find exactly 48 scene headings. Found {len(prompts)}. '
            f'Missing: {missing}; unexpected: {extra}. '
            'Expected headings like: ## Scene 1: Heading. '
            'Make sure the complete image-prompts.md document is pasted into the Prompts box. '
            f'First lines received:\n{preview}'
        )
    return [prompts[n] for n in range(1, 49)]

PROMPTS = parse_scene_prompts(prompt_box.value)
print(f'Validated {len(PROMPTS)} numbered prompts.')
print('Scene 01 preview:', PROMPTS[0][:350] + ('…' if len(PROMPTS[0]) > 350 else ''))

## 2. Load SDXL Base and the official Refiner
The Base model composes the image; the Refiner processes the final denoising stage to improve detail. Both are public Stability AI SDXL 1.0 checkpoints. This takes longer and uses more resources than Base alone.


In [ ]:
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline, EulerAncestralDiscreteScheduler
import torch
base_id = 'stabilityai/stable-diffusion-xl-base-1.0'
refiner_id = 'stabilityai/stable-diffusion-xl-refiner-1.0'
pipe = StableDiffusionXLPipeline.from_pretrained(
    base_id, torch_dtype=torch.float16, variant='fp16', use_safetensors=True
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.enable_attention_slicing()
pipe.enable_model_cpu_offload()
refiner = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    refiner_id, text_encoder_2=pipe.text_encoder_2, vae=pipe.vae,
    torch_dtype=torch.float16, use_safetensors=True, variant='fp16'
)
refiner.enable_vae_slicing()
refiner.enable_attention_slicing()
refiner.enable_model_cpu_offload()
print('SDXL Base + Refiner are loaded. Generation will take longer, but uses a two-stage quality workflow.')

## 3. Generate all 48 high-quality images
Click Run once. Each scene is generated at 1024×576 with 40 sampling steps; the Refiner handles the final 20% of denoising. Images are saved after each scene. If Colab disconnects but preserves the runtime, rerun the setup/model cells, paste the same prompts and rerun this cell; saved scene files are skipped.


In [ ]:
from tqdm.auto import tqdm
negative_prompt = 'text, letters, words, watermark, logo, signature, blurry, low quality, low detail, poorly composed, distorted anatomy, extra limbs, malformed hands, duplicate subjects, flat lighting, oversaturated, jpeg artifacts, plastic texture, muddy details'
style_suffix = ('Premium cinematic 3D CGI science documentary still, ultra-detailed professional visual development, physically based materials, realistic global illumination, nuanced volumetric lighting, rich but natural color grading, crisp focal subject, layered atmospheric depth, coherent scene design, polished film-quality rendering, wide 16:9 composition, no text or labels.')
for scene_number, prompt in enumerate(tqdm(PROMPTS, desc='Generating 48 refined scenes'), start=1):
    target = IMAGE_DIR / f'scene-{scene_number:02d}.webp'
    if target.exists() and target.stat().st_size > 1000:
        continue
    full_prompt = prompt.strip() + '\n\n' + style_suffix
    generator = torch.Generator(device='cpu').manual_seed(42000 + scene_number)
    with torch.inference_mode():
        latent_result = pipe(
            prompt=full_prompt, negative_prompt=negative_prompt,
            width=1024, height=576, num_inference_steps=40, guidance_scale=6.5,
            denoising_end=0.8, output_type='latent', generator=generator
        )
        refined_result = refiner(
            prompt=full_prompt, negative_prompt=negative_prompt,
            image=latent_result.images, num_inference_steps=40,
            denoising_start=0.8, guidance_scale=5.5
        )
    image = refined_result.images[0].convert('RGB')
    image.save(target, 'WEBP', quality=92, method=6)
    del latent_result, refined_result, image
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f'Saved refined scene {scene_number:02d}/48')
print('All available scene generations are complete.')

## 4. Validate and download the 48-image ZIP
The ZIP uses numbered WebP filenames and is designed for the repository's existing 48-image ZIP importer. The importer has a 25 MiB ZIP limit, so check the reported size before downloading.

In [ ]:
files = sorted(IMAGE_DIR.glob('scene-*.webp'))
if len(files) != 48:
    raise RuntimeError(f'Expected 48 images, found {len(files)}. Generate missing scenes before continuing.')
for number, path in enumerate(files, start=1):
    expected = f'scene-{number:02d}.webp'
    if path.name != expected or path.stat().st_size < 1000:
        raise RuntimeError(f'Invalid or missing image: {expected}')
zip_path = Path('/content/science-channel-48-images.zip')
def build_zip():
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_STORED) as archive:
        for path in files:
            archive.write(path, arcname=path.name)
build_zip()
size_mb = zip_path.stat().st_size / (1024 * 1024)
if size_mb > 25:
    print(f'ZIP is {size_mb:.1f} MiB; recompressing WebP images to quality 78 to meet the importer limit…')
    for path in files:
        with Image.open(path) as image:
            image.convert('RGB').save(path, 'WEBP', quality=78, method=6)
    build_zip()
    size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f'Validated 48 images. ZIP size: {size_mb:.1f} MiB')
if size_mb > 25:
    raise RuntimeError('The ZIP is still larger than the repository importer limit of 25 MiB. Do not upload it; the images need further compression.')
from google.colab import files
files.download(str(zip_path))

## 5. Send the ZIP to the existing render workflow
1. Download `science-channel-48-images.zip` above.
2. Open your GitHub repository on the `main` branch and upload the ZIP into `science-channel/telegram-uploads/` (create the folder if GitHub does not show it; normally it already exists).
3. Commit the ZIP to `main`.
4. Open **Actions → Telegram ZIP Import → Run workflow**.
5. For `episode_path`, copy the exact episode folder path from the caption of the Telegram prompt document. It looks like `science-channel/daily-episodes/YYYY-MM-DD/episode-001-topic-slug`.
6. For `zip_path`, enter `science-channel/telegram-uploads/science-channel-48-images.zip`.
7. Run the workflow. It validates/imports the images and the existing renderer takes over.

**Do not also send these images individually to the Telegram image receiver.** Use only one intake route for an episode.